# Registro, Gate de calidad y Promotion

Este notebook continúa el **mismo backend MLflow compartido** de Notebook 03. El entrenamiento produce runs reales; las decisiones usan exactamente `invoiceops.ml.gate` e `invoiceops.ml.registry`.

- Un **Run** registra una ejecución de experimento; un **Model Version** es un artefacto registrado desde un run elegible.
- Un **Gate** evalúa métricas; **Promotion** es una acción humana independiente y explícita.
- **Latest** es cronológico; no es automáticamente **Best**.
- `champion` es un alias móvil, no un Model Version reescrito.

`DEMO_ROOT` guarda solamente estado técnico idempotente. No es un backend MLflow, Registry ni fuente de datos operacional.

> **Orden y efecto lateral.** Ejecuta 03 antes de este notebook. Recursos: `DEMO_ROOT/state.json`, runs/artifacts y Model Versions/aliases en MLflow. Condición: el estado técnico se crea si falta; los runs se crean solo si no existen; Registry cambia solo en celdas `MODIFICA ESTADO`. Idempotencia: `state.json` registra acciones terminadas y reutiliza recursos existentes. Recuperación: relee estado y Registry, corrige el preflight y no repitas una promotion a ciegas. Las consultas y el Gate son de solo lectura. Al terminar una promotion, reinicia la Model API y confirma `/health`: Registry state no es Runtime state.

In [ ]:
# Side-effect contract: resource=var/local-demo/notebook-state/state.json; condition=missing local state; idempotency=existing state is reused; recovery=inspect or reset only the declared local-demo root.
import contextlib
import html
import importlib
import io
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from IPython.display import HTML, display
from mlflow.exceptions import MlflowException
from mlflow.tracking import MlflowClient

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
for import_path in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))
gate = importlib.import_module("invoiceops.ml.gate")
PRECISION_THRESHOLD = gate.PRECISION_THRESHOLD
RECALL_THRESHOLD = gate.RECALL_THRESHOLD

DEMO_ROOT = Path(
    os.environ.get("INVOICEOPS_NOTEBOOK_DEMO_ROOT", PROJECT_ROOT / "var" / "local-demo" / "notebook-state")
).resolve()
DEMO_ROOT.mkdir(parents=True, exist_ok=True)
STATE_PATH = DEMO_ROOT / "state.json"
TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI")
MLFLOW_UI_URL = os.environ.get("INVOICEOPS_MLFLOW_UI_URL")
if not TRACKING_URI:
    raise RuntimeError(
        "Preflight MLflow: falta MLFLOW_TRACKING_URI en este kernel. Inicia el servidor compartido, exporta la URI y reinicia el kernel antes de continuar."
    )
if not MLFLOW_UI_URL:
    raise RuntimeError(
        "Preflight MLflow: falta INVOICEOPS_MLFLOW_UI_URL en este kernel. Configura la URL del navegador y reinicia el kernel antes de continuar."
    )
state = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {}
state.setdefault("candidates", {})
state.setdefault("registered_versions", {})
state.setdefault("completed_actions", {})
state.setdefault("promotion_history", [])
mlflow.set_tracking_uri(TRACKING_URI)
try:
    MlflowClient().search_experiments(max_results=1)
except Exception as error:
    raise RuntimeError(
        f"Preflight MLflow: no se puede conectar a {TRACKING_URI}. Verifica que el único servidor MLflow compartido esté activo y que la URI sea accesible. Detalle: {error}"
    ) from error
print(f"Backend MLflow compartido accesible: {TRACKING_URI}")
print(f"Estado idempotente local: {STATE_PATH}")

## Antes de comparar: qué es un candidato

Un **candidato** es un run que estamos evaluando. No es todavía una versión registrada ni una decisión de producción.

- **A** reutiliza el run `random_forest` creado por 03. **B** es la única segunda ejecución independiente de `random_forest`, creada aquí solo si aún no existe: enseña que **Run != Model Version**.
- **C** reutiliza el run `logistic` de 03 y **D** el baseline `dummy` de 03. Si 03 no se ejecutó, este notebook crea explícitamente los candidatos faltantes mediante el CLI productivo en el mismo backend; nunca crea otro backend.
- A/B pueden tener métricas iguales por datos y seed deterministas, pero tienen `run_id` distinto y originan Model Versions distintas. D se observa; no se intenta promover.

In [ ]:
# Side-effect contract: resource=local state plus MLflow runs; condition=missing candidate; idempotency=reuses recorded or matching runs; recovery=inspect state and MLflow before retrying.
CANDIDATES = {"A": "random_forest", "B": "random_forest", "C": "logistic", "D": "dummy"}


def save_state():
    STATE_PATH.write_text(json.dumps(state, indent=2, sort_keys=True) + "\n")


def show_technical_messages(title, text):
    if text.strip():
        display(
            HTML(
                f"<details><summary>{html.escape(title)}</summary><pre>{html.escape(text)}</pre></details>"
            )
        )


def matching_runs(model_type):
    experiment = mlflow.get_experiment_by_name("invoice-risk")
    if experiment is None:
        return pd.DataFrame()
    return mlflow.search_runs(
        [experiment.experiment_id],
        filter_string=f"params.model_type = '{model_type}' and params.dataset_version = 'invoice-risk-v1'",
        order_by=["attributes.start_time DESC"],
    )


def run_training(model_type):
    environment = os.environ | {
        "MLFLOW_TRACKING_URI": TRACKING_URI,
        "PYTHONPATH": str(PROJECT_ROOT / "src"),
    }
    result = subprocess.run(
        [sys.executable, "-m", "invoiceops.ml.train", "--model", model_type],
        cwd=PROJECT_ROOT,
        env=environment,
        text=True,
        capture_output=True,
        check=False,
    )
    show_technical_messages(f"Mensajes técnicos de MLflow para {model_type}", result.stderr)
    if result.returncode:
        raise RuntimeError(
            f"El entrenamiento de {model_type} falló. Revise el bloque técnico mostrado."
        )
    runs = matching_runs(model_type)
    if runs.empty:
        raise RuntimeError(
            f"El CLI terminó sin registrar el candidato {model_type} en el backend compartido."
        )
    return str(runs.iloc[0]["run_id"])


client = MlflowClient()
for candidate, model_type in CANDIDATES.items():
    run_id = state["candidates"].get(candidate)
    if run_id is not None:
        try:
            existing_type = client.get_run(run_id).data.params.get("model_type")
            if existing_type != model_type:
                run_id = None
        except MlflowException:
            run_id = None
    if run_id is None:
        existing = matching_runs(model_type)
        if candidate != "B" and not existing.empty:
            run_id = str(existing.iloc[0]["run_id"])
            print(f"{candidate}: reutiliza run de Notebook 03 ({run_id}).")
        elif candidate == "B" and len(existing) >= 2:
            run_id = str(existing.iloc[1]["run_id"])
            print(f"B: reutiliza el segundo RandomForest existente ({run_id}).")
        else:
            run_id = run_training(model_type)
            origin = (
                "segundo RandomForest B"
                if candidate == "B"
                else "candidato faltante recuperado con CLI"
            )
            print(f"{candidate}: creado en el backend compartido ({origin}, {run_id}).")
        state["candidates"][candidate] = run_id
save_state()
print(
    "Preparación completada: A/C/D reutilizan 03 cuando existen; B es el único segundo RandomForest."
)

## Ver esta ejecución en MLflow

MLflow UI es un **visor del mismo backend compartido** que usa este notebook, no una copia. Inicia ese único servidor siguiendo el README antes de abrir Jupyter; no inicies un segundo servidor aquí.

En la misma UI revisa **Experiments -> invoice-risk -> runs** para evidencia de experimentación y **Models -> invoice-review -> v1/v2, challenger/champion** para Registry. La auditoría SQLite aparece recién en 05 y se consulta en sus tablas o en el portal, no como métrica de MLflow.

In [ ]:
print(f"Abre la UI de MLflow: {MLFLOW_UI_URL}")
print("Experiments -> invoice-risk -> runs; Models -> invoice-review -> versiones y aliases.")

## Comparar evidencia, no elegir automáticamente

La tabla y los gráficos muestran evidencia para una decisión humana. El pequeño desplazamiento de A/B en el scatter es **solo visual**: las métricas reales no cambian. Sirve para ver dos runs que ocupan exactamente las mismas coordenadas.

In [ ]:
rows = []
for candidate, model_type in CANDIDATES.items():
    run_id = state["candidates"][candidate]
    run = client.get_run(run_id)
    rows.append(
        {
            "Candidato": candidate,
            "Tipo": model_type,
            "Run ID": run_id,
            **run.data.params,
            **run.data.metrics,
            **run.data.tags,
        }
    )
comparison = pd.DataFrame(rows).sort_values("Candidato").reset_index(drop=True)
comparison["Run ID corto"] = comparison["Run ID"].str[:8]
comparison["Dataset"] = comparison["dataset_version"]
comparison["Accuracy"] = comparison["accuracy"]
comparison["Precision"] = comparison["precision"]
comparison["Recall"] = comparison["recall"]
comparison["F1"] = comparison["f1"]
comparison["ROC AUC"] = comparison["roc_auc"]
display(
    comparison[
        [
            "Candidato",
            "Tipo",
            "Run ID corto",
            "Dataset",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "ROC AUC",
        ]
    ].style.format(
        {name: "{:.3f}" for name in ["Accuracy", "Precision", "Recall", "F1", "ROC AUC"]}
    )
)

colors = {"random_forest": "#1f77b4", "logistic": "#ff7f0e", "dummy": "#7f7f7f"}
markers = {"random_forest": "o", "logistic": "s", "dummy": "X"}
offsets = {"A": (-0.006, 0.004), "B": (0.006, -0.004), "C": (0, 0), "D": (0, 0)}
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for _, row in comparison.iterrows():
    dx, dy = offsets[row["Candidato"]]
    axes[0].scatter(
        row["recall"] + dx,
        row["precision"] + dy,
        s=150,
        color=colors[row["Tipo"]],
        marker=markers[row["Tipo"]],
        label=row["Tipo"],
    )
    axes[0].annotate(
        f"{row['Candidato']} ({row['Tipo']})",
        (row["recall"] + dx, row["precision"] + dy),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=10,
    )
axes[0].axvline(RECALL_THRESHOLD, color="#555555", linestyle="--", label="umbral recall")
axes[0].axhline(PRECISION_THRESHOLD, color="#555555", linestyle=":", label="umbral precision")
axes[0].set(
    xlabel="Recall",
    ylabel="Precisión",
    title="Precision vs recall (A/B desplazados solo para verlos)",
)
handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(dict(zip(labels, handles)).values(), dict(zip(labels, handles)).keys(), loc="best")
metric_table = comparison.set_index("Candidato")[["Accuracy", "Precision", "Recall"]].T
metric_table.plot(kind="bar", ax=axes[1], color=["#1f77b4", "#1f77b4", "#ff7f0e", "#7f7f7f"])
axes[1].set(title="Métricas por candidato", ylabel="Valor", xlabel="Métrica", ylim=(0, 1))
axes[1].legend(title="Candidato")
fig.tight_layout()
plt.show()

## Quality Gate: elegibilidad medible

**Qué ocurre:** el Gate consulta las métricas persistidas de cada run. No entrena ni registra nada.

**Qué se ejecuta:** `invoiceops.ml.gate.evaluate_run`, con los umbrales productivos `recall >= 0.18` y `precision >= 0.48`.

**Qué comprobar:** A/B deben aprobar en este dataset determinista y D debe fallar por recall 0. C se interpreta con su resultado real: si falla, no es elegible; si aprobara, seguiría sin Promotion automática.

In [ ]:
from invoiceops.ml.gate import PRECISION_THRESHOLD, RECALL_THRESHOLD, evaluate_run

gate_rows = []
for _, row in comparison.iterrows():
    recall, precision = evaluate_run(row["Run ID"])
    status = "PASS" if recall >= RECALL_THRESHOLD and precision >= PRECISION_THRESHOLD else "FAIL"
    if row["Candidato"] in {"A", "B"}:
        interpretation = (
            "Run independiente de random_forest: mismas métricas deterministas, identidad distinta."
        )
    elif row["Candidato"] == "D":
        interpretation = f"Baseline observado: accuracy {row['Accuracy']:.3f}, pero recall 0; no detecta positivos."
    elif status == "FAIL":
        interpretation = f"Run válido, no elegible: recall {recall:.3f} queda bajo el umbral {RECALL_THRESHOLD:.2f}."
    else:
        interpretation = (
            "Run válido y elegible; un PASS no ejecuta Registration ni Promotion automáticamente."
        )
    gate_rows.append(
        {
            "Candidato": row["Candidato"],
            "Tipo": row["Tipo"],
            "Run ID corto": row["Run ID"][:8],
            "Dataset": row["Dataset"],
            "Recall": recall,
            "Precision": precision,
            "Gate": status,
            "Interpretación": interpretation,
        }
    )
gates = pd.DataFrame(gate_rows)
display(gates.style.format({"Recall": "{:.3f}", "Precision": "{:.3f}"}))
assert (gates.set_index("Candidato").loc[["A", "B"], "Gate"] == "PASS").all()
assert gates.set_index("Candidato").at["D", "Gate"] == "FAIL"
print(
    f"Resultado real: {(gates['Gate'] == 'PASS').sum()} PASS y {(gates['Gate'] == 'FAIL').sum()} FAIL."
)

## Registry: registrar A y B para separar Run de Version

**Qué ocurre:** se registran únicamente A/B, los dos runs `random_forest` que aprobaron. Cada Registration crea o recupera una versión dinámica y actualiza `challenger`.

**Qué se ejecuta:** `invoiceops.ml.registry.register_model`. **⚠ MODIFICA ESTADO** del Registry en el backend MLflow compartido.

**Qué comprobar:** comparar el estado antes/después: A y B tienen `Run ID` distinto y `vN` distinto aunque sus métricas sean iguales. Reejecutar la celda no crea otra versión; la acción queda registrada.

In [ ]:
# Side-effect contract: resource=MLflow Model Versions, challenger alias, and local state; condition=A/B pass Gate; idempotency=completed action is reused; recovery=inspect Registry and local state, never create a duplicate version blindly.
from invoiceops.ml.registry import MODEL_NAME, promote_model, register_model
from notebooks.demo_helpers import run_mutable_action_once


def alias_version(alias):
    try:
        return str(client.get_model_version_by_alias(MODEL_NAME, alias).version)
    except MlflowException:
        return None


def registry_table():
    rows = []
    for version in client.search_model_versions(f"name='{MODEL_NAME}'"):
        aliases = [
            alias
            for alias in ("challenger", "champion")
            if alias_version(alias) == str(version.version)
        ]
        run = client.get_run(version.run_id)
        rows.append(
            {
                "Versión": f"v{version.version}",
                "Run ID corto": version.run_id[:8],
                "Tipo": run.data.params.get("model_type", "desconocido"),
                "Aliases": ", ".join(aliases) or "sin alias",
                "Creada (UTC)": pd.to_datetime(
                    version.creation_timestamp, unit="ms", utc=True
                ).strftime("%Y-%m-%d %H:%M UTC"),
            }
        )
    return pd.DataFrame(
        rows, columns=["Versión", "Run ID corto", "Tipo", "Aliases", "Creada (UTC)"]
    )


print("Estado antes de Registration")
display(registry_table())
for candidate in ("A", "B"):
    if gates.set_index("Candidato").at[candidate, "Gate"] != "PASS":
        raise RuntimeError(f"El candidato {candidate} no aprobó el Quality Gate.")
    technical_stderr = io.StringIO()
    with contextlib.redirect_stderr(technical_stderr):
        version = str(
            run_mutable_action_once(
                f"register-{candidate}",
                state["completed_actions"],
                lambda candidate=candidate: register_model(state["candidates"][candidate]),
            )
        )
    show_technical_messages(
        f"Mensajes técnicos de Registry al registrar {candidate}", technical_stderr.getvalue()
    )
    state["registered_versions"][candidate] = version
save_state()
print("Estado después de Registration")
display(registry_table())
print("Reversibilidad: una versión creada no se borra aquí; el alias sí puede moverse después.")

## Promotion: mover un alias, no reescribir una versión

**Qué ocurre:** la decisión explícita mueve `champion` A -> B -> A -> B para mostrar alias switching.

**Qué se ejecuta:** `invoiceops.ml.registry.promote_model`. **⚠ MODIFICA ESTADO** del Registry en el backend MLflow compartido.

**Qué comprobar:** el historial dice desde qué `vN` hasta qué `vN`; el primer origen es `sin champion`. Es reversible mover el alias, pero ninguna Model Version se borra.

In [ ]:
# Side-effect contract: resource=MLflow champion alias and local promotion history; condition=explicit eligible A/B action; idempotency=completed action is reused; recovery=inspect alias/history and move the alias deliberately.
def promote_step(action, candidate):
    if gates.set_index("Candidato").at[candidate, "Gate"] != "PASS":
        raise RuntimeError("Promotion bloqueada: el candidato no aprobó el Quality Gate.")
    target = str(state["registered_versions"][candidate])

    def promote_once():
        previous = alias_version("champion")
        if previous != target:
            promote_model(target)
        state["promotion_history"].append(
            {
                "Paso": len(state["promotion_history"]) + 1,
                "Acción": f"promote {candidate}",
                "Desde": f"v{previous}" if previous else "sin champion",
                "Hacia": f"v{target}",
            }
        )
        return target

    return run_mutable_action_once(action, state["completed_actions"], promote_once)


print("Estado antes de Promotion")
display(registry_table())
for action, candidate in [
    ("promote-A", "A"),
    ("promote-B", "B"),
    ("switch-to-A", "A"),
    ("final-champion-B", "B"),
]:
    technical_stderr = io.StringIO()
    with contextlib.redirect_stderr(technical_stderr):
        promote_step(action, candidate)
    show_technical_messages(
        f"Mensajes técnicos de Promotion: {action}", technical_stderr.getvalue()
    )
save_state()
display(pd.DataFrame(state["promotion_history"], columns=["Paso", "Acción", "Desde", "Hacia"]))
print("Estado después de Promotion")
display(registry_table())
print(
    f"Champion actual: v{alias_version('champion')}. Cambiar este alias es reversible; las versiones permanecen."
)

## C y D: bloqueo y baseline

**Qué ocurre:** C se somete al mismo control antes de Promotion. D queda explícitamente como baseline observado: nunca se registra ni se intenta promover.

**Qué se ejecuta:** la guarda de elegibilidad previa a `promote_model`; solo si C falló, la operación se bloquea antes de mutar el Registry.

**Qué comprobar:** C fallido deja un mensaje de bloqueo claro. Si sus métricas reales cambiasen y aprobara, el notebook no lo promueve automáticamente: requeriría una decisión humana separada.

In [ ]:
c_gate = gates.set_index("Candidato").at["C", "Gate"]
if c_gate == "FAIL":
    try:
        promote_step("promote-C", "C")
    except RuntimeError as error:
        print(error)
    print("C sigue siendo un run válido, pero no es elegible para Registration ni Promotion.")
else:
    print(
        "C aprobó el Gate real. No se registra ni promueve automáticamente: falta una decisión humana explícita."
    )
assert "D" not in state["registered_versions"]
print("D es el baseline observado: no se registra ni se intenta promover.")